# Module 5: Regression using Artificial Neural Networks

<i>Instructor: Gustavo Sganzerla Martinez</i>

# Lab


In [ ]:
## Imports
from sklearn.datasets import load_diabetes
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

We will use the pre-loaded diabetes dataset, available in sklearn.datasets. 
Additional information from this dataset can be found at https://scikit-learn.org/stable/datasets/toy_dataset.html (9.1.2)

In [ ]:
diabetes = load_diabetes()

###lets inspect it
diabetes.feature_names

###lets create a dataframe to structure the dataset for better vis

df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)


In total, there are 442 observations (rows) and 10 features (columns)

In addition to the features age, sex, bmi (body-mass index), and bp (average blood pressure), we have the following:

    s1 = total serum cholesterol
    s2 = low-density lipoproteins (ldl, bad cholesterol)
    s3 = high-density lipoproteins (hdl, good cholesterol)
    s4 = total cholesteral ratio
    s5 = log of serum triglycerides
    s6 = blood glucose level (target)


In [ ]:
###now lets only visualize one feature
###we use only one representative feature (bmi) 
###lets try to predict the target (s6; blood glucose level) with it

X = diabetes.data[:, np.newaxis, 2]
y = diabetes.target


plt.figure(figsize=(6,5))
plt.scatter(X, y, alpha=0.7)

plt.xlabel("BMI")
plt.ylabel("blood glucose level")

plt.show()

###note that the points are scattered
###how would you summarize this prediction with a single prediction rule (line)?

In [ ]:
###we can observe the looks of a positive correlation between bmi and the target
###we can even quantify this relationship with a simple correlation (pearson) test
r, p = pearsonr(
    diabetes.data[:,2],
    diabetes.target
)

print(r)
print(p)

###we can assume positive correlation
###BMI and the target are linearly associated and there is enough statistical evidence for it

In [ ]:
###lets add a linear regression here

###train and test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

###model instantiating
model = LinearRegression()

###model fitting
model.fit(X_train, y_train)

###visualization
plt.figure(figsize=(6,5))
plt.scatter(X_train, y_train, alpha=0.7)
plt.plot(X_test, model.predict(X_test), color="red", linewidth=3)

plt.xlabel("BMI")
plt.ylabel("Blood glucose level")
plt.title("Linear regression")

plt.show()

Now, let's evaluate the performance of a linear regression model using BMI to predict blood glucose.

Let's consider the example:

<table>
    <thead>
        <tr>
            <th>Patient</th>
            <th>Actual</th>
            <th>Predicted</th>
            <th>Error</th>
        </tr>
    </thead>
    <tbody>
    <tr>
        <td>A</td>
        <td>200</td>
        <td>180</td>
        <td>-20</td>
    </tr>
    <tr>
        <td>B</td>
        <td>140</td>
        <td>110</td>
        <td>+30</td>
    </tr>
    <tr>
        <td>C</td>
        <td>115</td>
        <td>115</td>
        <td>0</td>
    </tr>
    </tbody>
</table>
    

<b>Metrics to evaluate regression performance</b>

<table>
  <thead>
    <tr>
      <th>Metric</th>
      <th>Question</th>
      <th>Interpretation</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><b>MAE</b><br>(Mean Absolute Error)</td>
      <td>How wrong are we on average?</td>
      <td>Typical prediction error. It represents the average distance between predictions and true values.</td>
    </tr>
    <tr>
      <td><b>RMSE</b><br>(Root Mean Squared Error)</td>
      <td>How bad are large mistakes?</td>
      <td>Similar to MAE, but gives more importance to large prediction errors.</td>
    </tr>
    <tr>
      <td><b>R²</b><br>(Coefficient of Determination)</td>
      <td>How much variation do we explain?</td>
      <td>Measures the proportion of variability in the outcome explained by the model.</td>
    </tr>
  </tbody>
</table>

In [ ]:
###let's evaluate on the test split

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.2f}")

In [ ]:
###instead of only using the BMI as a predictor, let's use all the data

###variables
X = diabetes.data
y = diabetes.target

##train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
###model fit
model.fit(X_train, y_train)

###evaluate
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.2f}")

###lower MAE means the predictions are closer to the observed target
###lower RMSE means fewer larger prediction errors
###higher r^2 means the model explains more of the variation in the outcome

###adding more information improved the model, but there is still a lot of unexplained variation

We need to consider that a linear regression model assumes the relationship between the inputs and outcome can be describe using a 
linear combination of features.

Most biological systems are non-linear.

now the question becomes:
    how can we improve our prediction?
        1. more data
        2. better features
        3. a more complex model (ANNs)


In [ ]:
###data prep for the ANN

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
###feature scaling (very important for the ANN)
###the coefficients were mathematically solved
###any major differences in scale will bias the ANN

scaler = StandardScaler()

###scale only based on the train to avoid leakage
X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

An objet of the class <b>Dense</b> indicates a layer in the ANN.
<b>Dense</b> takes:
Number of neurons: 32, 16, 1

input_shape: Number of features considered. We second index of the X (rows, columns); always in the first layer of the ANN.

activation: 'relu', this is the activation function of each neuron. ReLu (rectified linear unit) outputs the actual value if positive, and 0 if negative. Introduces non-linearity.

The last <b>Dense</b> layer specifies a single neuron as the output.

In [ ]:
###instantiating the model
model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)), ###32 neurons learning the combination of inputs
    Dense(16, activation='relu'), ###16 neurons learning the combination of inputs
    Dense(1)
])

In [ ]:
###compile the model
model.compile(
    optimizer='adam', ###how weights are updated ###adaptive moment estimation
    loss='mse', ##what the model tries to minimize
    metrics=['mae'] #performance
)


In [ ]:
###train the model
history = model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=100
)

In [ ]:
###this shows the ANN is learning, see the decrease in MSE after each epoch of training

plt.plot(history.history["loss"], label="Training")
plt.plot(history.history["val_loss"], label="Validation")

plt.xlabel("Epoch")
plt.ylabel("MSE")

In [ ]:
###now lets evaluate the same way we did with the linear regression
y_pred_ann = model.predict(X_test_scaled).flatten()

mae = mean_absolute_error(y_test, y_pred_ann)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_ann))
r2 = r2_score(y_test, y_pred_ann)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.2f}")

In conclusion, the ANN did not outperform a linear regression. This means the problem may not require a highly flexible model or the network might require more data and optimization

We should always attain to the principle of parsimony (the simpler the better)

We can try to adjust the ANN to see if the performance will improve, at this stage, you can toy around with the layers Dense(16, activation='relu') to either 
    a simpler (less layers, less neurons)
    or 
    a more complex (more layers, more neurons)

We can also adjust the training time (epochs)
    epochs = 100

For each test, we should plot the loss (train and val) and also evaluate the ANN on the test data

In [ ]:
##an important design decision is to add an early stopping instead of guessing the number of epochs
###here, the trainin will stop once the loss (val_loss) stops decreasing

##lets define the stopping criteria
early_stop = EarlyStopping(
    monitor="val_loss", ###what we are monitoring
    patience=30, ###number of epochs allowed
    restore_best_weights=True ###if the model gets the weights of the last epoch or the best epoch (i.e., lower val_loss)
)


###build the model again
model_ES = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)), ###32 neurons learning the combination of inputs
    Dense(16, activation='relu'), ###32 neurons learning the combination of inputs
    Dense(1)
])

###compile the model
model_ES.compile(
    optimizer='adam', ###how weights are updated
    loss='mse', ##what the model tries to minimize
    metrics=['mae'] #performance
)

##now fit with earlystopping
history_ES = model_ES.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=200,
    callbacks=[early_stop]
)

###plot 
plt.plot(history_ES.history["loss"], label="Training")
plt.plot(history_ES.history["val_loss"], label="Validation")

plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.show()

##evaluate
y_pred_ann = model_ES.predict(X_test_scaled).flatten()

mae = mean_absolute_error(y_test, y_pred_ann)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_ann))
r2 = r2_score(y_test, y_pred_ann)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.2f}")

In [ ]:
###another important design decision is to add a dropout layer (regularization technique to reduce overfit)
###think of a team project where two students are super smart and the success of the project is always dependent on them
###by adding a dropout layer, we send random neurons to each training epoch (e.g. 0.2)

###instantiate
model_dropout = Sequential([
    Dense(32, activation="relu",
          input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1)
])

##compile

model_dropout.compile(
    optimizer='adam', ###how weights are updated
    loss='mse', ##what the model tries to minimize
    metrics=['mae'] #performance
)

##train

history = model_dropout.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=100,
    callbacks=[early_stop],
    verbose=0
)

##view loss

plt.plot(history.history["loss"], label="Training")
plt.plot(history.history["val_loss"], label="Validation")

plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.show()

##evaluate

y_pred_ann = model_dropout.predict(X_test_scaled).flatten()

mae = mean_absolute_error(y_test, y_pred_ann)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_ann))
r2 = r2_score(y_test, y_pred_ann)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.2f}")

In conclusion

Linear regression = 
    MAE: 42.79
    RMSE: 53.85
    R²: 0.45

ANN (32, 16), 100 epochs = 
    MAE: 53.62
    RMSE: 65.34
    R²: 0.19

ANN (32, 16) + early stopping = 
    MAE: 54.75
    RMSE: 66.36
    R²: 0.17

ANN (32, 16) + early stopping + dropout = 
    MAE: 53.33
    RMSE: 65.21
    R²: 0.20


ML is not about using the most complext algorithm, it is about chosing the appropriate model for the problem

An ANN is a more flexible model, but at a cost

If the dataset does not contain enough non-linearity, a simpler model can outperform an ANN

In [ ]:
###now, lets try to simulate a non-linear problem

np.random.seed(42)

X = np.linspace(-3, 3, 300).reshape(-1,1)

noise = np.random.normal(0, 1, size=X.shape)

y = X**2 + noise

In [ ]:
##visualize

plt.figure(figsize=(6,5))

plt.scatter(X, y, alpha=0.6)

plt.xlabel("X")
plt.ylabel("y")
plt.title("Synthetic nonlinear relationship")

plt.show()

In [ ]:
###linear model
linear_model = LinearRegression()

linear_model.fit(X, y)

y_pred = linear_model.predict(X)

In [ ]:
##plot the linear model
plt.figure(figsize=(6,5))

plt.scatter(X, y, alpha=0.6)

plt.plot(
    X,
    y_pred,
    color="red",
    linewidth=3,
    label="Linear regression"
)

plt.xlabel("X")
plt.ylabel("y")
plt.legend()

plt.show()

In [ ]:
###calculate performance
rmse = np.sqrt(mean_squared_error(y, y_pred))
r2 = r2_score(y, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.2f}")

In [ ]:
###now introduce an ANN

###split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

###build the ANN
ann = Sequential([
    Dense(16, activation="relu", input_shape=(1,)),
    Dense(16, activation="relu"),
    Dense(1)
])

##compile

ann.compile(
    optimizer="adam",
    loss="mse",
    metrics=['mae'] #performance
)


In [ ]:
###train
history = ann.fit(
    X_train,
    y_train,
    epochs=100,
    verbose=0
)

In [ ]:
###evaluate
y_ann = ann.predict(X).flatten()

plt.figure(figsize=(6,5))

plt.scatter(X, y, alpha=0.5)

plt.plot(
    X,
    y_ann,
    color="red",
    linewidth=3,
    label="ANN"
)

plt.xlabel("X")
plt.ylabel("y")
plt.legend()

plt.show()

print(
    "Linear R²:",
    r2_score(y, y_pred)
)

print(
    "ANN R²:",
    r2_score(y, y_ann)
)